# Fluorescence decay and lifetime analysis

This notebook is the notebook-style entry point for tttrlib fluorescence-decay workflows. It explains how decay histograms, instrument response functions (IRFs), background patterns, phasor summaries, and the `Fit23`-`Fit26` model family fit together.

Use the higher-level helper or class APIs for normal analysis. Use `DecayFitData` and `DecayFitXX.fit` only when you need exact control over the arrays passed to the C++ layer or when you are testing a low-level integration.

## Core data model

A decay fit uses a binned microtime histogram rather than raw photon timestamps.

- `data`: measured decay counts.
- `irf`: instrument response function with the same binning as `data`.
- `background`: background or scatter pattern with the same shape as `data`.
- `dt`: width of one microtime bin.
- `period`: excitation period, in the same units as lifetimes.
- `fixed`: integer mask where `0` means optimized and `1` means fixed.

Polarization-resolved fits use **Jordi format**: the parallel decay followed by the perpendicular decay in one one-dimensional array. `irf` and `background` must use the same stacked order.

## Which fit should I use?

| Goal | Helper | Class | Low-level function | Parameter vector | Typical fixed mask | Main example |
| --- | --- | --- | --- | --- | --- | --- |
| Single lifetime with polarization-resolved decay and anisotropy/rotation output | `fit23` | `Fit23` | `DecayFit23.fit` | `[tau, gamma, r0, rho]` | `[0, 1, 1, 0]` when fitting lifetime and rotation time | :doc:`../auto_examples/fluorescence_decay/plot_fit23` |
| Bi-exponential lifetime model without anisotropy | `fit24` | `Fit24` | `DecayFit24.fit` | `[tau1, gamma, tau2, A2, offset]` | `[0, 0, 0, 0, 0]` for a full synthetic fit; fix offset when known | :doc:`../auto_examples/fluorescence_decay/plot_fit24` |
| Choose the best lifetime from four fixed candidates | `fit25` | `Fit25` | `DecayFit25.fit` | `[tau1, tau2, tau3, tau4, gamma, r0]` | `[0, 0, 0, 0, 1, 1]` for candidate selection only | :doc:`../auto_examples/fluorescence_decay/plot_fit25` |
| Fit a mixture fraction of two known reference patterns | `fit26` | `Fit26` | `DecayFit26.fit` | `[fraction_1]` | `[0]` | :doc:`../auto_examples/fluorescence_decay/plot_fit26` |

The combined helper overview is available at :doc:`../auto_examples/fluorescence_decay/plot_fit_helpers`.

## Result dictionary contract

The high-level helpers and classes return a dictionary:

- `x`: fit-specific parameter/result array. Some slots are outputs written by the low-level routine, so interpret it with the fit-specific table above.
- `fixed`: the fixed/free mask used for the fit.
- `twoIstar`: MLE quality value. Smaller values indicate a better model for the same data and assumptions.
- `model`: fitted model decay, present only when `include_model=True`.

Always plot `data`, `irf`, `background`, and `model` together when validating a new fit setup.

## Minimal synthetic setup

The following cells create a small deterministic Jordi-format IRF and low-count decay. They are intentionally tiny so the notebook can be downloaded and run without external TTTR data.

In [ ]:
import numpy as np
import tttrlib

n_channels = 32
period = 32.0
time_axis = np.linspace(0.0, period, n_channels * 2)
irf = (
    np.exp(-0.5 * ((time_axis - 2.0) / 0.25) ** 2)
    + np.exp(-0.5 * ((time_axis - 18.0) / 0.25) ** 2)
).astype(np.float64)
background = np.zeros_like(irf)
dt = time_axis[1] - time_axis[0]

low_count_decay = np.array([
    0, 0, 0, 1, 9, 7, 5, 5, 5, 2, 2, 0, 0, 0, 0, 0,
    1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 3, 2, 2, 2, 2, 3, 0, 1, 0, 1, 1, 1, 2,
    0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
], dtype=np.int32)

## One-call helper path

Use helpers when one decay is being fitted in a notebook or script and the shared instrument settings do not need to be reused many times.

In [ ]:
fit23_result = tttrlib.fit23(
    data=low_count_decay,
    irf=irf,
    background=background,
    dt=dt,
    period=period,
    tau=2.2,
    gamma=0.01,
    r0=0.38,
    rho=1.2,
    include_model=True,
)

print(f"fit23 tau = {fit23_result['x'][0]:.3f}")
print(f"fit23 twoIstar = {fit23_result['twoIstar']:.3f}")

## Reusable class path

Use a class when many decays share the same IRF, background, bin width, and excitation period. This is common for FLIM pixels and burst-level decay batches.

In [ ]:
fit23 = tttrlib.Fit23(
    dt=dt,
    irf=irf,
    background=background,
    period=period,
    g_factor=1.0,
    l1=0.1,
    l2=0.1,
    convolution_stop=len(irf) // 2 - 1,
)
class_result = fit23(
    data=low_count_decay,
    initial_values=np.array([2.2, 0.01, 0.38, 1.2]),
    fixed=np.array([0, 1, 1, 0], dtype=np.int16),
    include_model=True,
)
print(f"class tau = {class_result['x'][0]:.3f}")

## Low-level `DecayFitData` path

`DecayFitData` is the shared container used by the low-level `DecayFit23`-`DecayFit26` routines. Prefer helpers or classes for normal user-facing notebooks; low-level calls are mainly useful for exact integration tests, migration from legacy wrappers, and advanced custom pipelines.

In [ ]:
params = tttrlib.DecayFitData(
    dt=dt,
    corrections=np.array([period, 1.0, 0.1, 0.1, len(irf) // 2 - 1]),
    irf=irf,
    background=background,
    data=low_count_decay,
)

x = np.zeros(8, dtype=np.float64)
x[:4] = [2.2, 0.01, 0.38, 1.2]
fixed = np.array([0, 1, 1, 0], dtype=np.int16)
two_istar = tttrlib.DecayFit23.fit(x, fixed, params)
print(f"low-level tau = {x[0]:.3f}")
print(f"low-level twoIstar = {two_istar:.3f}")

## Phasor and mean-lifetime summaries

Phasor and moment-based lifetime summaries are useful before fitting because they show whether a pixel or selection behaves like one population, a mixture, or poor signal. Use the FLIM examples for image-level workflows:

- :doc:`../auto_examples/flim/plot_phasor`
- :doc:`../auto_examples/flim/plot_mean_lifetime`
- :doc:`../auto_examples/flim/plot_lifetime_moments`
- :doc:`../auto_examples/flim/plot_mle_lifetime`

For fit model details, see :doc:`../fit-guide`.

## Troubleshooting checklist

- If fitted lifetimes use the wrong units, check that `dt`, `period`, and initial lifetime values are in the same units.
- If the model is shifted relative to the data, inspect IRF alignment and `convolution_stop`.
- If `Fit24` swaps lifetimes or drives `A2` to a boundary, the two-component model may not be identifiable from the photon counts.
- If `Fit25` always chooses the same candidate, check that the four candidate lifetimes cover the observed decay.
- If `Fit26` returns an implausible mixture fraction, normalize and compare the two reference patterns before fitting.
- If low-level calls behave differently from helper calls, compare `DecayFitData.corrections`, Jordi ordering, and `fixed` arrays.